In [1]:
from block import mine_block
from blockchain import Blockchain, make_next_block

In [2]:
DIFFICULTY = 3

def section(title: str) -> None:
    print(f"\n-* {title} *-")
    
def build_valid_chain() -> Blockchain:
    bc = Blockchain()
    bc.append_block(
        make_next_block(bc, [{"sender": "Phenyo", "recipient": "Thato", "amount": 10}])
    )
    bc.append_block(
        make_next_block(bc, [{"sender": "Thato", "recipient": "Phenyo", "amount": 4}])
    )
    return bc
    
def main() -> None:
    # *Checkpoint 1: append_block() does not allow bad candidates to be added to the chain*
    section("Checkpoint 1: append_block() writer-protection gates")
    bc = build_valid_chain()
    tip_before, length_before = bc.tip().hash, len(bc)

    from block import Block 

    bad_link = Block(
        index=length_before, timestamp=0, transactions=[{"x": 1}],
        previous_hash="deadbeef" * 8,
    )
    try:
        bc.append_block(bad_link)
        raise AssertionError("expected ValueError for bad previous_hash")
    except ValueError as exc:
        print("rejected bad previous_hash:", exc)

    bad_index = Block(index=99, timestamp=0, transactions=[{"x": 1}], previous_hash=tip_before)
    try:
        bc.append_block(bad_index)
        raise AssertionError("expected ValueError for bad index")
    except ValueError as exc:
        print("rejected bad index:", exc)

    liar = make_next_block(bc, [{"x": 1}])
    liar.hash = "ff" * 32
    try:
        bc.append_block(liar)
        raise AssertionError("expected ValueError for bad stored hash")
    except ValueError as exc:
        print("rejected bad stored hash:", exc)

    assert len(bc) == length_before
    assert bc.tip().hash == tip_before
    assert bc.verify_chain() is True
    print(f" our original blockchain unchanged:still valid, length {len(bc)}")

    #*Checkpoint 2: valid chain passes verification*
    section("Checkpoint 2: valid chain (length >= 3)")
    ledger = build_valid_chain()
    print(f"length: {len(ledger)}")
    for blk in ledger.chain:
        print(f"  {blk}  prev={blk.previous_hash[:12]}...")
    print(f"verify_chain() -> {ledger.verify_chain()}")
    assert len(ledger) == 3
    assert ledger.verify_chain() is True

    #*Checkpoint 3: using verify_chain to see what happens when we alter the data of an intermediate block*
    section("Checkpoint 3: tamper with Block 1's transaction data")
    old_hash_1 = ledger.chain[1].hash
    print(f"Before: Block 1 transactions = {ledger.chain[1].transactions}")
    ledger.chain[1].transactions[0]["amount"] = 999
    print(f"After:  Block 1 transactions = {ledger.chain[1].transactions}")
    print(f"verify_chain() -> {ledger.verify_chain()}")
    assert ledger.verify_chain() is False
    assert ledger.chain[1].hash == old_hash_1  # stored digest has not updated yet
    assert ledger.chain[1].hash != ledger.chain[1].compute_hash()
    print("pillar 2 (self-hash) fails: stored hash != recomputation for Block 1")

    #*Checkpoint 4: showing how re-hashing only one block along the chain is not enough*
    section("Checkpoint 4: recalculate only Block 1's hash")
    ledger.chain[1].hash = ledger.chain[1].compute_hash()
    print(f"verify_chain() -> {ledger.verify_chain()}")
    assert ledger.verify_chain() is False
    assert ledger.chain[2].previous_hash == old_hash_1
    assert ledger.chain[2].previous_hash != ledger.chain[1].hash
    print("pillar 3 (link) fails: Block 2.previous_hash is stale")
    print(f"  Block 2.previous_hash: {ledger.chain[2].previous_hash[:16]}...")
    print(f"  Block 1.hash now:      {ledger.chain[1].hash[:16]}...")

    #*Checkpoint 5: difficulty rule points out a cheap suffix rewrite*
    section("Checkpoint 5: suffix rewrite without proof-of-work")
    for i in range(1, len(ledger.chain)):
        if i > 1:
            ledger.chain[i].previous_hash = ledger.chain[i - 1].hash
        ledger.chain[i].hash = ledger.chain[i].compute_hash()

    print(f"verify_chain() [no difficulty]      -> {ledger.verify_chain()}")
    assert ledger.verify_chain() is True  # the links repaired and the hashes for each block in the chain are self-consistent
    assert ledger.chain[1].transactions[0]["amount"] == 999  # we see fraud is still in the data
    print("If we dont assign an adequate difficulty rule, the cheap rewrite is accepted")
    print("the fraudulent amount is now taken as 'valid' history in our blockchain.")

    print(f"verify_chain(difficulty={DIFFICULTY}) -> {ledger.verify_chain(difficulty=DIFFICULTY)}")
    assert ledger.verify_chain(difficulty=DIFFICULTY) is False
    print("When we enforece an adequate difficulty rule, the same repaired chain we saw before FAILS the verification:")
    print("the rewritten hashes were never re-mined, so they don't have the")
    print(f"required {DIFFICULTY} leading zero hex digits. Fixing this would")
    print("require making the effort to actually re-mine every block from the tamper point to")
    print("the tip, this involves real, difficulty-scaled computational work, not just")
    print("recomputing a hash.")

    # This section shows the kind of effort that re-mining that one block would take, for concreteness.
    remined = ledger.chain[1]
    mine_block(remined, difficulty=DIFFICULTY)
    print(f"\n(For reference: re-mining Block 1 alone at difficulty={DIFFICULTY} "
          f"found nonce={remined.nonce}, hash={remined.hash[:16]}...)")

    print("\nAll checkpoints behaved as expected.")


if __name__ == "__main__":
    main()



-* Checkpoint 1: append_block() writer-protection gates *-
rejected bad previous_hash: bad previous_hash: does not match chain tip
rejected bad index: bad index: expected 3, got 99
rejected bad stored hash: stored hash does not match recomputation
 our original blockchain unchanged:still valid, length 3

-* Checkpoint 2: valid chain (length >= 3) *-
length: 3
  Block(index=0, hash=120a20ac98da...)  prev=000000000000...
  Block(index=1, hash=26a218b030cc...)  prev=120a20ac98da...
  Block(index=2, hash=dffd39624dc8...)  prev=26a218b030cc...
verify_chain() -> True

-* Checkpoint 3: tamper with Block 1's transaction data *-
Before: Block 1 transactions = [{'sender': 'Phenyo', 'recipient': 'Thato', 'amount': 10}]
After:  Block 1 transactions = [{'sender': 'Phenyo', 'recipient': 'Thato', 'amount': 999}]
verify_chain() -> False
pillar 2 (self-hash) fails: stored hash != recomputation for Block 1

-* Checkpoint 4: recalculate only Block 1's hash *-
verify_chain() -> False
pillar 3 (link) fail